# Metric Evaluation

Figures for the analysis layer, driven from the eval outputs that run_evaluation.py writes. Sample-level scores are read from merged.csv for the curves / quadrant / embeddings. If the output CSVs aren't present the tables are recomputed on the fly.

Run run_evaluation.py first (per dataset / pooled set)

eval_df = df[df["binder"].astype(str).isin(["0", "1"])].copy()
eval_df["binder"] = eval_df["binder"].astype(int)

design_df = df[df["type"] == "design"].copy()

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from config.datasets import cfg
from config.paths import RAW_DATA_DIR, EVALUATION_DIR, METRIC_YAML
from config.analysis import COLORS_MAP
from preprocessing.align import load_metric_directions
from analysis.distributions import get_numeric_metrics
from analysis.evaluation import (
    calculate_all_metrics, compute_auroc_pvalue, cliffs_delta, cohens_d,
    spearman_correlation, label_metrics, pca_embedding, umap_embedding,
    plot_top_metrics_bar, plot_roc_curves, plot_pr_curves, plot_precision_ranking,
    plot_metric_comparison_grid, plot_top_metrics_bar_by_dataset, plot_quadrant, plot_embedding, plot_spearman_heatmap,
)
# Uniform figure styling: one validated palette and recessive axes/grid for every figure
# in the project (config/palette.py). Call once, before plotting.
from config.palette import apply_plot_style
apply_plot_style()
# Auto-save every figure below to data/figures/<dataset>/ at print resolution, named after the
# plot and its key arguments (e.g. top_metrics_bar_pr_auc.png). An explicit save_path= still wins.
# Comment this out to go back to display-only.
from config.paths import FIGURES_DIR
from analysis.figures import set_figure_dir
set_figure_dir(FIGURES_DIR / cfg.name)

In [ ]:
base = RAW_DATA_DIR / cfg.name
eval_dir = EVALUATION_DIR / cfg.name          # where run_evaluation.py wrote the tables

# Labeled evaluation split only (binder in {0,1}); designs ("?") are excluded.
# eval.csv is written by prep for both single datasets and pooled sets.
merged_path = base / "merged.csv"  # df with eval + designs
df = pd.read_csv(merged_path)
#eval_df = df[df["binder"].astype(str).isin(["0", "1"])].copy()
#eval_df["binder"] = eval_df["binder"].astype(int)

#design_df = df[df["binder"] == "?"].copy()

design_df = base / "design.csv"

eval_path = base / "eval.csv"
eval_df = pd.read_csv(eval_path) if eval_path.exists() else split_eval_design(df)[0]

In [ ]:
metrics = get_numeric_metrics(eval_df)
directions = load_metric_directions(METRIC_YAML)

# Prefer the run_evaluation.py outputs; recompute only if a table is missing.
def load_or_compute(name, fn):
    p = eval_dir / name
    return pd.read_csv(p) if p.exists() else fn()

rankings = load_or_compute("rankings.csv", lambda: calculate_all_metrics(eval_df, metrics, directions))
rankings.head(15)

## Rankings

In [ ]:
plot_top_metrics_bar(rankings, show=True)

In [ ]:
plot_top_metrics_bar(rankings, score="pr_auc", show=True)    # PR-AUC


In [ ]:
plot_top_metrics_bar(rankings, score="ap_norm_ds_mean", show=True)

In [ ]:
plot_roc_curves(eval_df, rankings, directions, show=True)

In [ ]:
plot_pr_curves(eval_df, rankings, directions, show=True)

In [ ]:
plot_precision_ranking(rankings, show=True)

In [ ]:
plot_metric_comparison_grid(rankings, show=True)

## Quadrant (top-2 metrics)

In [ ]:
plot_quadrant(df, rankings, COLORS_MAP, show=True)
rankings.head(15)

### Interactive quadrant (Plotly)

In [ ]:
import plotly.graph_objects as go

marker_map_ply = {0: "diamond-x", "0": "diamond-x", 1: "circle", "1": "circle", "?": "square" }
alpha_map = {"dms": 0.4, "original": 0.4, "design": 1.0, "alanine_scan": 0.4}

top = rankings.sort_values("pr_auc", ascending=False).head(20)
m1i, m2i = top.iloc[1], top.iloc[2]
m1, m2 = m1i["metric"], m2i["metric"]
t1, t2 = m1i["opt_threshold_raw"], m2i["opt_threshold_raw"]

pdf = df.groupby(["source", "sample", "type", "binder"])[[m1, m2]].agg(["mean", "std"]).reset_index()
pdf.columns = [f"{c[0]}_{c[1]}" if c[1] else c[0] for c in pdf.columns]

fig = go.Figure()
for t, tdf in pdf.groupby("type"):
    op = alpha_map.get(t, 1.0)
    for (source, binder), sub in tdf.groupby(["source", "binder"]):
        fig.add_trace(go.Scatter(
            x=sub[f"{m1}_mean"], y=sub[f"{m2}_mean"], mode="markers",
            marker=dict(color=COLORS_MAP.get(source), symbol=marker_map_ply[binder], size=8, opacity=op),
            name=f"{source} | {t} | binder={binder}", text=sub["sample"],
            hovertemplate="<b>%{text}</b><br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
        ))
fig.add_vline(x=t1, line_dash="dash", line_color="black", opacity=0.3)
fig.add_hline(y=t2, line_dash="dash", line_color="black", opacity=0.3)
fig.update_layout(width=900, height=700, plot_bgcolor="white", title=f"{m1} vs {m2}", xaxis_title=m1, yaxis_title=m2)
fig.show()

## Top separators (from distinct metrics families)

In [ ]:
from analysis.evaluation.metrics import load_metric_families
from analysis.evaluation import top_metrics_distinct_family, plot_class_distributions

# Top separating metrics, one per YAML family
families = load_metric_families(METRIC_YAML)
top_fam = top_metrics_distinct_family(rankings, families, top_n=5)

# Distributions include the Design class, so plot the full merged table
# (the eval df above is label-only and excludes designs).
plot_class_distributions(
    df, rankings, metrics=top_fam["metric"].tolist(),
    save_path=eval_dir / "top5_family_distinct_distributions.png", show=True,
)

#df.head(30)

## Effect sizes

In [ ]:
cliff = load_or_compute("cliffs_delta.csv", lambda: cliffs_delta(df, metrics, directions))
auroc = load_or_compute("auroc_pvalue.csv", lambda: compute_auroc_pvalue(df, metrics, directions)) #--> secondary
#cohen = load_or_compute("cohens_d.csv", lambda: cohens_d(df, metrics, directions)) --> unnecessary
cliff.head(15)

## Redundancy

In [ ]:
# model-set metrics only: the filter-only ones (energy / developability / sequence) are never
# ranked, so their redundancy is not the question here. rankings.csv is sorted by PR-AUC, and its
# plain top-20 pulls in 4 of them.
top20 = rankings.loc[rankings["in_model_set"]].head(20)["metric"].tolist()
plot_spearman_heatmap(spearman_correlation(df, top20), show=True)

In [ ]:
# all 46 model-set metrics: spearman_correlation defaults to model_set_only=True, so the
# filter-only ones (energy / developability / sequence) are excluded without naming them.
plot_spearman_heatmap(spearman_correlation(df), show=True)

In [ ]:
emb_pca, names, var = pca_embedding(df)
print(f"PC1 {var[0]:.1%}  PC2 {var[1]:.1%}")
plot_embedding(emb_pca, names, label_metrics(names, "family"), xlabel=f"PC1 ({var[0]:.1%})", ylabel=f"PC2 ({var[1]:.1%})", show=True)
plot_embedding(emb_pca, names, label_metrics(names, "model"), xlabel=f"PC1 ({var[0]:.1%})", ylabel=f"PC2 ({var[1]:.1%})", show=True)

In [ ]:
emb_umap, names = umap_embedding(df)
plot_embedding(emb_umap, names, label_metrics(names, "family"), xlabel="UMAP 1", ylabel="UMAP 2", show=True)
plot_embedding(emb_umap, names, label_metrics(names, "model"), xlabel="UMAP 1", ylabel="UMAP 2", show=True)

## Per-dataset breakdown (pooled sets)

For a pooled set, `run_evaluation.py` also writes `rankings_by_dataset.csv`. Each dataset is
ranked and baselined on its OWN rows, so a panel is read against its own chance line.

Raw `pr_auc` / `precision_at_pct` bar HEIGHTS are not comparable between datasets: their baseline
is each dataset's prevalence (0.09 snir to 0.74 benoit). `ap_norm` = (PR-AUC - prevalence) /
(1 - prevalence) is the comparable view (0 = no-skill, 1 = perfect).

In [ ]:
p = eval_dir / "rankings_by_dataset.csv"
if p.exists():
    by_ds = pd.read_csv(p)
    model_set = rankings.loc[rankings["in_model_set"], "metric"]
    for score in ("ap_norm", "pr_auc", "precision_at_pct"):
        plot_top_metrics_bar_by_dataset(by_ds, score=score, model_set=model_set, show=True)
else:
    print("No per-dataset breakdown (single dataset, or run_evaluation not run on a set).")